# Lesson 5: Jev Use Cases in Code

One worked example for each category from Lesson 04, using public datasets and generated games:

1. **AI-powered workflows:** route support tickets; escalate the unsure ones to an LLM.
2. **Map-reducing big data:** label every commit in a public repo's git history.
3. **Real-time applications:** Jev plays text adventure games, one action per turn.
4. **Verify everything:** check scientific claims against the abstracts they cite.

Set up the repo first by following **Setup** in the root `README.md`. Populate the repository-root `.env`; the first code cell loads it. Every section needs `TYPESAFE_API_KEY`. Section 1's escalation step also needs `OPENAI_API_KEY`.

The data is downloaded or generated when you run the notebook, into `data/` next to this notebook and the Hugging Face cache. Costs use each API's reported token usage at standard uncached USD rates, checked 2026-10-07. Latency is wall-clock time from your machine, so it includes the network round trip.

## Setup

`ask_all` sends the same questions about many inputs, `CONCURRENCY` calls at a time. Jev has no batch endpoint, so this is how you map over a dataset.

TypeSafe publishes rate limits of 80 requests/s and 100K tokens/s ([docs.typesafe.ai/models](https://docs.typesafe.ai/models), checked 2026-10-07). At ~150 ms per call, 12 concurrent calls is about 80 requests/s. A semaphore caps calls in flight, not requests per second: faster calls can still exceed the published limit. The SDK retries `429` responses, honoring `Retry-After`; a `TypeSafeRateLimitError` that survives the retries is raised.

In [36]:
import os
from pathlib import Path

from dotenv import load_dotenv

env_file = Path("../.env")  # notebook kernels start in the lesson folder
if not env_file.is_file():
    raise FileNotFoundError("Create .env in the course repository root. See README.md Setup.")
load_dotenv(env_file, override=True)
missing_keys = [key for key in ("TYPESAFE_API_KEY", "OPENAI_API_KEY") if not os.environ.get(key)]
if missing_keys:
    raise RuntimeError(f"Populate {', '.join(missing_keys)} in the repository-root .env before running.")

import asyncio
import statistics
import time

import pandas as pd
from typesafe_sdk import AsyncTypeSafeClient, Choice, Noul, RetryPolicy, TypeSafeClient

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

CONCURRENCY = 12
JEV_USD_PER_INPUT_TOKEN = 0.042 / 1_000_000  # outputs are free


def jev_cost(responses) -> float:
    return sum(r.usage.input_tokens for r in responses) * JEV_USD_PER_INPUT_TOKEN


async def ask_all(states, questions):
    """Ask the same questions about every state. Returns (responses, latencies in ms)."""
    semaphore = asyncio.Semaphore(CONCURRENCY)

    async with AsyncTypeSafeClient(retry=RetryPolicy(max_retries=5)) as client:
        async def ask(state):
            async with semaphore:
                start = time.perf_counter()
                response = await client.system_one(state=state, questions=questions)
                return response, (time.perf_counter() - start) * 1000

        start = time.perf_counter()
        results = await asyncio.gather(*(ask(state) for state in states))
        wall_s = time.perf_counter() - start

    responses = [response for response, _ in results]
    latencies = [ms for _, ms in results]
    print(f"{len(states):,} Jev calls in {wall_s:.1f} s | "
          f"median {statistics.median(latencies):.0f} ms per call | "
          f"{sum(r.usage.input_tokens for r in responses):,} input tokens | ${jev_cost(responses):.4f}")
    return responses, latencies


def threshold_table(df, correct, confidence, thresholds=(0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99)):
    """For each confidence threshold: how much is handled automatically, and how accurately."""
    rows = []
    for threshold in thresholds:
        kept = df[df[confidence] >= threshold]
        rows.append({
            "threshold": threshold,
            "handled automatically": f"{len(kept) / len(df):.0%}",
            "samples": len(kept),
            "accuracy": f"{kept[correct].mean():.1%}",
            "errors": int((~kept[correct]).sum()),
        })
    return pd.DataFrame(rows)

## 1. AI-Powered Workflows: Route Support Tickets

Modeled on the support-triage projects in Lesson 04. Jev picks a queue for each ticket and flags status requests. Code routes the confident ones and escalates the rest to GPT-6 Sol.

**Data:** the [Bitext customer support dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) (CDLA-Sharing-1.0): 26,872 synthetic customer messages, each labeled with one of 11 categories and 27 intents. The labels let us measure queue accuracy, including for tickets with reply actions. This is a controlled dataset, not a production inbox.

In [2]:
from datasets import load_dataset

N_TICKETS = 500

tickets = (
    load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
    .shuffle(seed=0)
    .select(range(N_TICKETS))
    .to_pandas()[["instruction", "category", "intent"]]
)
tickets.head()

/Users/alex/pro/jev-intro-to-decision-models/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,instruction,category,intent
0,there are errors setting up a delivery address,SHIPPING,set_up_shipping_address
1,i want assistance seeing ur accepted payment o...,PAYMENT,check_payment_methods
2,I have got to cancel the newsletter subscription,SUBSCRIPTION,newsletter_subscription
3,can ya help me to change to the gold account,ACCOUNT,switch_account
4,I am waiting for a compensation of {{Currency ...,REFUND,track_refund


In [3]:
tickets

,instruction,category,intent
0,there are errors setting up a delivery address,SHIPPING,set_up_shipping_address
1,i want assistance seeing ur accepted payment o...,PAYMENT,check_payment_methods
2,I have got to cancel the newsletter subscription,SUBSCRIPTION,newsletter_subscription
3,can ya help me to change to the gold account,ACCOUNT,switch_account
4,I am waiting for a compensation of {{Currency ...,REFUND,track_refund
...,...,...,...
495,"I have lost bill #12588, can you mail it to me?",INVOICE,get_invoice
496,I need assistance modifying the damn data on m...,ACCOUNT,edit_account
497,i do not know what to do to inform of a paymen...,PAYMENT,payment_issue
498,i need assistance with creating a freemium acount,ACCOUNT,create_account


### The Jev call: one `Choice` and one `Noul` per ticket

The queues are the dataset's 11 categories. Each description summarizes the intents in that category.

In [4]:
QUEUES = {
    "ACCOUNT": "Create, edit, switch, or delete an account; registration and password problems",
    "CANCEL": "Cancellation fees and penalties",
    "CONTACT": "Reach customer service or a human agent",
    "DELIVERY": "Delivery options and how long delivery takes",
    "FEEDBACK": "Complaints, claims, and reviews",
    "INVOICE": "Find or check an invoice",
    "ORDER": "Place, change, cancel, or track an order",
    "PAYMENT": "Payment methods and payment problems",
    "REFUND": "Refund policy, request a refund, or track a refund",
    "SHIPPING": "Set up or change a shipping address",
    "SUBSCRIPTION": "Newsletter subscription",
}

TICKET_QUESTIONS = {
    "queue": Choice(
        instructions="Which support queue should handle this message?",
        criteria=QUEUES,
    ),
    "status_request": Noul(
        instructions="Is the customer asking for the status of an order or refund they already have?",
        criteria={
            "true": "Wants to track an order, a shipment, or a refund already requested",
            "false": "Anything else: new requests, changes, cancellations, policies, fees, complaints",
        },
    ),
}

### The code that acts on the answers

Branch order matters: the confidence check runs first, so a low-confidence ticket reaches escalation before the reply branch. This demo returns action labels; it does not send replies or look up tracking links. `queue.confidence` summarizes the Choice distribution, while `status_request.noul` is the probability of the status-request proposition.

In [5]:
AUTO_ROUTE = 0.90  # queue confidence needed to act without an LLM
AUTO_REPLY = 0.90  # status_request probability needed to send a tracking link


def route(answers) -> str:
    queue = answers["queue"]
    if queue.confidence < AUTO_ROUTE:
        return "escalate"                      # unsure: an LLM reads it
    if answers["status_request"].noul >= AUTO_REPLY:
        return f"auto_reply:{queue.choice}"    # send the order or refund tracking link
    return f"queue:{queue.choice}"             # confident: route it

In [6]:
responses, latencies = await ask_all(tickets["instruction"].tolist(), TICKET_QUESTIONS)

tickets["queue"] = [r.answers["queue"].choice for r in responses]
tickets["confidence"] = [r.answers["queue"].confidence for r in responses]
tickets["status_request"] = [r.answers["status_request"].noul for r in responses]
tickets["action"] = [route(r.answers) for r in responses]
tickets["correct"] = tickets["queue"] == tickets["category"]
ticket_jev_cost = jev_cost(responses)

tickets["action"].str.split(":").str[0].value_counts()

500 Jev calls in 4.9 s | median 108 ms per call | 291,645 input tokens | $0.0122


action
queue         419
escalate       44
auto_reply     37
Name: count, dtype: int64

In [7]:
handled = tickets[tickets["action"] != "escalate"]
replies = tickets[tickets["action"].str.startswith("auto_reply")]
is_status = tickets["intent"].isin(["track_order", "track_refund"])

print(f"Routed without a human: {len(handled) / len(tickets):.0%} of tickets, "
      f"{handled['correct'].mean():.1%} to the right queue")
print(f"Automatic replies: {len(replies)}, "
      f"{replies['intent'].isin(['track_order', 'track_refund']).mean():.0%} were status requests")
print(f"Status requests in the sample: {is_status.sum()}, "
      f"{replies.index.isin(tickets[is_status].index).sum()} answered automatically")
print(f"Accuracy if we trusted every answer: {tickets['correct'].mean():.1%}")

Routed without a human: 91% of tickets, 98.9% to the right queue
Automatic replies: 37, 86% were status requests
Status requests in the sample: 41, 32 answered automatically
Accuracy if we trusted every answer: 97.2%


### Confidence and queue accuracy

For each threshold, measure how many tickets are handled automatically and how accurately their queues are labeled. Useful confidence ranking does not establish calibration; we check probabilities against observed accuracy in section 4. Choose a threshold on labeled data, then confirm coverage and accuracy on held-out examples. Queue accuracy and reply quality are separate measures.

In [8]:
threshold_table(tickets, correct="correct", confidence="confidence").rename(columns={"accuracy": "queue accuracy"})

,threshold,handled automatically,samples,queue accuracy,errors
0,0.00,100%,500,97.2%,14
1,0.50,99%,495,97.8%,11
2,0.70,97%,485,97.9%,10
3,0.80,95%,475,98.7%,6
4,0.90,91%,456,98.9%,5
5,0.95,85%,426,99.3%,3
6,0.99,76%,381,100.0%,0


In [10]:
pd.set_option("display.max_colwidth", 90)
tickets[~tickets["correct"]].sort_values("confidence")[["instruction", "category", "intent", "queue", "confidence"]]

,instruction,category,intent,queue,confidence
403,help me to check the early exit few,CANCEL,check_cancellation_fee,CONTACT,0.30
216,can you help me to earn some of your product?,ORDER,place_order,CONTACT,0.34
435,where could I see how soon can I expect my shipment?,DELIVERY,delivery_period,ORDER,0.47
412,how to see if there is anything new on my compensation,REFUND,track_refund,PAYMENT,0.53
65,need assistance seeing when my shipment is gonna arrive,DELIVERY,delivery_period,ORDER,0.74
156,is it possible to ordet from {{Delivery Country}},DELIVERY,delivery_options,ORDER,0.74
328,what do i have to do to see when will my parcel arrive,DELIVERY,delivery_period,ORDER,0.75
161,can I see when will my article arrive?,DELIVERY,delivery_period,ORDER,0.79
366,I do not know what to do to see when will my purchase arrive,DELIVERY,delivery_period,ORDER,0.89
96,is it possible to see the withdrawal charges?,CANCEL,check_cancellation_fee,PAYMENT,0.94


### Escalate the unsure tickets to an LLM

GPT-6 Sol reads each escalated ticket and picks a queue: one structured-output call per ticket, with reasoning effort `none`. Sol is the same model Lesson 03 benchmarks against.

In [11]:
from typing import Literal

import openai
from pydantic import BaseModel

SOL_USD_PER_TOKEN = {"input": 2.00 / 1_000_000, "output": 10.00 / 1_000_000}  # verified 2026-10-12


class LLMRouting(BaseModel):
    queue: Literal[tuple(QUEUES)]  # one of the QUEUES keys


llm = openai.AsyncOpenAI()  # reads OPENAI_API_KEY
llm_semaphore = asyncio.Semaphore(8)


async def route_with_llm(ticket: str) -> dict:
    async with llm_semaphore:
        start = time.perf_counter()
        response = await llm.responses.parse(
            model="gpt-6-sol",
            input=f"Route this support message to one queue.\n\nQueues: {QUEUES}\n\nMessage: {ticket}",
            text_format=LLMRouting,
            reasoning={"effort": "none"},
        )
        latency_ms = (time.perf_counter() - start) * 1000
    if response.status != "completed":
        raise RuntimeError(f"GPT-6 Sol stopped with status {response.status}")
    usage = response.usage
    return {
        "llm_queue": response.output_parsed.queue,
        "llm_ms": latency_ms,
        "llm_usd": usage.input_tokens * SOL_USD_PER_TOKEN["input"]
        + usage.output_tokens * SOL_USD_PER_TOKEN["output"],
    }


escalated = tickets[tickets["action"] == "escalate"].copy()
llm_results = await asyncio.gather(*(route_with_llm(t) for t in escalated["instruction"]))
escalated = escalated.join(pd.DataFrame(llm_results, index=escalated.index))

print(f"{len(escalated)} escalated tickets | GPT-6 Sol median {escalated['llm_ms'].median():,.0f} ms "
      f"| ${escalated['llm_usd'].sum():.4f}")
print(f"On these tickets: Jev's top choice {escalated['correct'].mean():.0%} right, "
      f"GPT-6 Sol {(escalated['llm_queue'] == escalated['category']).mean():.0%} right")

44 escalated tickets | GPT-6 Sol median 1,378 ms | $0.0274
On these tickets: Jev's top choice 80% right, GPT-6 Sol 86% right


### The whole pipeline vs one LLM call per ticket

The Sol-on-all cost is an estimate extrapolated from the escalated tickets; we do not run Sol on all 500. The latency comparison uses the median for each model. Jev answers two questions per ticket, while Sol only picks a queue. A small escalation sample does not establish which model is better.

In [12]:
pipeline_correct = handled["correct"].sum() + (escalated["llm_queue"] == escalated["category"]).sum()
pipeline_cost = ticket_jev_cost + escalated["llm_usd"].sum()

print(f"Pipeline accuracy: {pipeline_correct / len(tickets):.1%} "
      f"({len(handled)} routed by Jev, {len(escalated)} by GPT-6 Sol)")
print("Cost per 1,000 tickets:")
print(f"  Jev on every ticket + GPT-6 Sol on escalations: ${pipeline_cost / len(tickets) * 1000:.3f}")
print(f"  GPT-6 Sol on every ticket (estimated):          ${escalated['llm_usd'].mean() * 1000:.3f}")
print("Median latency per ticket:")
print(f"  Jev: {statistics.median(latencies):,.0f} ms | GPT-6 Sol: {escalated['llm_ms'].median():,.0f} ms")

Pipeline accuracy: 97.8% (456 routed by Jev, 44 by GPT-6 Sol)
Cost per 1,000 tickets:
  Jev on every ticket + GPT-6 Sol on escalations: $0.079
  GPT-6 Sol on every ticket (estimated):          $0.622
Median latency per ticket:
  Jev: 108 ms | GPT-6 Sol: 1,378 ms


## 2. Map-Reducing Big Data: Label a Repo's Git History

Modeled on [commit-miner](https://github.com/devanshbatham/commit-miner). Map: ask Jev the same questions about every commit. Reduce: count, group, and filter the answers with pandas.

**Data:** up to 5,000 non-merge commits of [`psf/requests`](https://github.com/psf/requests) (Apache-2.0). A blobless clone downloads commits and file trees but no file contents, so the recorded clone took about a second. A cached clone preserves its existing snapshot. Each commit becomes its message plus the paths it changed. commit-miner also reads the diffs; we skip them to keep the clone small.

In [3]:
import subprocess

REPO_URL = "https://github.com/psf/requests.git"
REPO_DIR = DATA_DIR / "requests.git"
MAX_COMMITS = 5000  # requests has ~4,900 non-merge commits

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", "--bare", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)

log = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "--no-merges", f"--max-count={MAX_COMMITS}",
     "--date=short", "--format=%x00%h%x1f%ad%x1f%B%x1e", "--name-only"],
    capture_output=True, text=True, check=True,
).stdout

commits = []
for record in log.split("\x00")[1:]:
    header, files = record.split("\x1e")
    sha, date, message = header.split("\x1f")
    commits.append({"sha": sha, "date": date, "message": message.strip(), "files": files.strip().splitlines()})

commits = pd.DataFrame(commits)
print(f"{len(commits):,} commits from {commits['date'].min()} to {commits['date'].max()}")
commits.head()

4,883 commits from 2011-02-13 to 2026-09-21


,sha,date,message,files
0,611c6162,2026-09-21,Bump the actions group with 3 updates (#7628)\...,[.github/workflows/codeql-analysis.yml]
1,dae7ef63,2026-09-02,Bump https://github.com/astral-sh/ruff-pre-com...,[.pre-commit-config.yaml]
2,5460f467,2026-08-24,Update lock-issues.yml to latest dependencies ...,[.github/workflows/lock-issues.yml]
3,6af0b941,2026-08-24,Bump https://github.com/astral-sh/ruff-pre-com...,[.pre-commit-config.yaml]
4,8f8b212d,2026-08-17,Bump https://github.com/astral-sh/ruff-pre-com...,[.pre-commit-config.yaml]


### The Jev call: what kind of change, and is it a security fix?

A `Choice` gives each commit one kind, so we can group by it. commit-miner uses independent `Noul`s instead, because one commit can be several things at once.

In [4]:
COMMIT_QUESTIONS = {
    "kind": Choice(
        instructions="What kind of change is this commit? Judge `message` and the changed `files`.",
        criteria={
            "bug_fix": "Fixes incorrect behavior",
            "feature": "Adds new behavior or options",
            "refactor": "Restructures or cleans up code without changing behavior",
            "docs": "Documentation only",
            "tests": "Tests only",
            "build": "Dependencies, CI, packaging, or releases",
        },
    ),
    "security_fix": Noul(
        instructions=(
            "Does this commit fix or harden a security weakness "
            "that could affect confidentiality, integrity, or availability?"
        ),
        criteria={
            "true": "Fixes a vulnerability or strengthens a security control",
            "false": "An ordinary bug fix, feature, refactor, docs, tests, or build change",
        },
    ),
}

### Map

In [7]:
states = [{"message": c.message, "files": c.files} for c in commits.itertuples()]
responses, _ = await ask_all(states, COMMIT_QUESTIONS)

commits["kind"] = [r.answers["kind"].choice for r in responses]
commits["security_fix"] = [r.answers["security_fix"].noul for r in responses]
commits["year"] = commits["date"].str[:4]

4,883 Jev calls in 44.6 s | median 96 ms per call | 2,464,492 input tokens | $0.1035


In [9]:
commits

,sha,date,message,files,kind,security_fix,year
0,611c6162,2026-09-21,Bump the actions group with 3 updates (#7628)\...,[.github/workflows/codeql-analysis.yml],build,0.33,2026
1,dae7ef63,2026-09-02,Bump https://github.com/astral-sh/ruff-pre-com...,[.pre-commit-config.yaml],build,0.15,2026
2,5460f467,2026-08-24,Update lock-issues.yml to latest dependencies ...,[.github/workflows/lock-issues.yml],build,0.27,2026
3,6af0b941,2026-08-24,Bump https://github.com/astral-sh/ruff-pre-com...,[.pre-commit-config.yaml],build,0.15,2026
4,8f8b212d,2026-08-17,Bump https://github.com/astral-sh/ruff-pre-com...,[.pre-commit-config.yaml],build,0.14,2026
...,...,...,...,...,...,...,...
4878,c51b44d9,2011-02-13,ISC License by mee,[LICENSE],docs,0.02,2011
4879,75b499dd,2011-02-13,sample test fixtures (none),[test_requests.py],tests,0.05,2011
4880,04770187,2011-02-13,easy setup.py,[setup.py],build,0.05,2011
4881,d0bf5538,2011-02-13,no mo of that,[README],docs,0.10,2011


### Reduce: what kind of work happened each year?

These counts summarize Jev's predicted labels, not human-verified change types. The current year is incomplete.

In [10]:
pd.crosstab(commits["year"], commits["kind"], margins=True)

kind,bug_fix,build,docs,feature,refactor,tests,All
year,,,,,,,
2011,183,141,375,161,159,87,1106
2012,258,139,239,82,106,91,915
2013,172,42,153,38,44,60,509
2014,87,32,115,16,36,23,309
2015,64,38,109,13,26,38,288
2016,58,44,158,11,22,68,361
2017,52,179,177,11,56,25,500
2018,18,34,121,2,13,7,195
2019,12,22,202,7,4,2,249


### Reduce: find the security fixes

The regex checks each full message, case-insensitively. Jev judges whether the message and file paths suggest a security fix or hardening; it does not inspect diffs here. Treat flagged commits as candidates to inspect, not verified fixes. There are no gold security labels, so this is not a precision/recall benchmark.

In [13]:
SECURITY_AT = 0.8
KEYWORDS = r"secur|CVE|vulnerab|leak"

commits["first_line"] = commits["message"].str.split("\n").str[0]
commits["keyword_match"] = commits["message"].str.contains(KEYWORDS, case=False)
flagged = commits[commits["security_fix"] >= SECURITY_AT].sort_values("security_fix", ascending=False)

print(f"Jev flags {len(flagged)} commits as security fixes (probability >= {SECURITY_AT}); "
      f"{(~flagged['keyword_match']).sum()} of them never match /{KEYWORDS}/")
flagged[["sha", "date", "security_fix", "keyword_match", "first_line"]]

Jev flags 11 commits as security fixes (probability >= 0.8); 5 of them never match /secur|CVE|vulnerab|leak/


,sha,date,security_fix,keyword_match,first_line
752,3331e2ae,2018-06-28,0.95,True,Strip Authorization header whenever root URL c...
187,c0813a2d,2024-03-03,0.92,False,Use TLS settings in selecting connection pool
434,0a760ff0,2020-02-26,0.91,True,updating pipfile.lock for bleach security issu...
278,eb07385b,2023-01-12,0.90,True,ci: add minimum GitHub token permissions for w...
427,bfb93d4b,2020-04-03,0.89,True,update bleach security vulnerability
96,514b3f23,2026-02-11,0.87,True,Bump step-security/harden-runner from 2.13.0 t...
116,7618dbef,2025-07-28,0.82,True,Bump step-security/harden-runner from 2.12.0 t...
2504,8b7fcfb4,2013-09-28,0.82,False,Don't send the full URL on CONNECT messages.
2745,7d217bf9,2013-03-21,0.82,False,- Pull credentials out of the URL when possible.
260,74ea7cf7,2023-05-22,0.81,False,Merge pull request from GHSA-j8r2-6x86-q33q


In [15]:
# The other direction: keyword matches that Jev did not flag
commits[commits["keyword_match"] & (commits["security_fix"] < SECURITY_AT)].sort_values(
    "security_fix", ascending=False
)[["sha", "date", "security_fix", "first_line"]].head(15)

,sha,date,security_fix,first_line
2267,b361a4c0,2014-04-02,0.76,Make note of the fact that we fixed some CVEs
128,5b4b64c3,2025-06-05,0.67,Add more tests to prevent regression of CVE 20...
3407,1de14e6a,2012-04-25,0.41,fix usage of compat.urlparse
288,bda7f017,2022-06-29,0.41,Fixed warnings message to suggest using secure...
3202,3c0b9404,2012-07-17,0.39,address connection leak issue from #520
62,a044b020,2026-04-15,0.35,Move DigestAuth hash algorithms to use usedfor...
2133,958845ae,2014-09-04,0.31,requests[security]
1454,9ec34784,2016-11-16,0.29,Change setuptools extra for new secure extra.
4485,32a861b2,2011-08-09,0.29,Fixed memory leak (see http://bugs.python.org/...
2029,ff56e431,2015-01-28,0.22,Update support documentation to be more accurate


## 3. Real-Time Applications: Jev Plays Text Adventure Games

Modeled on [minecraft-agent](https://github.com/rmalde/minecraft-agent) and [jev-ultrafast](https://github.com/browser-use/jev-ultrafast): every turn, the environment lists the actions available, Jev picks one as a `Choice`, and code executes it.

**Data:** [TextWorld](https://github.com/microsoft/TextWorld) (Microsoft Research, MIT license) cooking games, like those in the First TextWorld Problems competition. Find the cookbook, collect the ingredients, cut and cook them as the recipe says, then prepare and eat the meal. The games are generated on your machine.

There's no planner model here. Once Jev reads the cookbook, code keeps the recipe in `state` for every later turn. Without that, Jev picks plausible actions with no plan, like eating a raw ingredient.

In [17]:
import textworld
import textworld.challenges

N_GAMES = 10
MAX_STEPS = 50
LATENCY_BUDGET_MS = 300  # per decision
GAME_DIR = DATA_DIR / "textworld"
GAME_DIR.mkdir(exist_ok=True)

_, make_cooking_game, _ = textworld.challenges.CHALLENGES["tw-cooking"]


def build_game(seed: int) -> Path:
    """A cooking game: 2 ingredients to find across 6 rooms, with doors, cutting, and cooking."""
    path = GAME_DIR / f"cooking-{seed}.z8"
    if not path.exists():
        options = textworld.GameOptions()
        options.seeds = seed
        options.path = str(path)
        settings = {"recipe": 2, "take": 2, "go": 6, "open": True, "cook": True, "cut": True,
                    "drop": False, "recipe_seed": seed, "split": None}
        textworld.generator.compile_game(make_cooking_game(settings, options), options)
    return path


games = [build_game(seed) for seed in range(1, N_GAMES + 1)]

### The Jev call: pick the next command

The game's admissible commands become the `Choice` options, so Jev can only pick a command the game accepts. Accepted commands can still repeat actions, waste turns, or lose the game.

Two details matter a lot. `clean` strips the game's status bar from each observation, and the instructions carry three rules for the slips Jev made without them: taking the wrong ingredient, and cooking one twice (which burns it and loses the game). When I ran these 10 games on 2026-10-06, a one-line instruction without the rules won 4; this version won 9. Both the observation and instructions changed, and one run on ten games does not establish a general win rate. Your runs can differ.

In [20]:
INFOS = textworld.EnvInfos(admissible_commands=True, objective=True, inventory=True, max_score=True)
client = TypeSafeClient()

ACTION_INSTRUCTIONS = (
    "Which command makes the most progress toward `goal`? "
    "Once `recipe` has been read, follow its directions in order. "
    "Take only the exact ingredients `recipe` lists; open containers to find missing ones. "
    "Never cook or cut the same ingredient twice."
)


def clean(text: str) -> str:
    """Drop the game's prompt and status bar, which follow the last '>'."""
    return text.split("\n>")[0].strip()


def next_action(game_state, recipe: str | None, history: list[str]):
    commands = [c for c in game_state.admissible_commands if c not in ("look", "inventory")]
    state = {
        "goal": game_state.objective,
        "recipe": recipe or "not read yet",
        "observation": clean(game_state.feedback),
        "inventory": game_state.inventory.strip(),
        "recent_actions": history[-8:],
    }
    start = time.perf_counter()
    response = client.system_one(state=state, questions={"action": Choice(
        instructions=ACTION_INSTRUCTIONS,
        criteria={command: None for command in commands},
    )})
    latency_ms = (time.perf_counter() - start) * 1000
    return response.answers["action"], latency_ms, response

### The loop: observe, decide, act

In [21]:
def play(path: Path, verbose: bool = False) -> dict:
    env = textworld.start(str(path), request_infos=INFOS)
    game_state = env.reset()
    recipe, history, latencies, responses = None, [], [], []
    if verbose:
        print(f"Objective: {game_state.objective}\n")

    for step in range(1, MAX_STEPS + 1):
        action, latency_ms, response = next_action(game_state, recipe, history)
        game_state, score, done = env.step(action.choice)
        if action.choice == "examine cookbook":
            recipe = clean(game_state.feedback)  # code keeps the plan
        history.append(action.choice)
        latencies.append(latency_ms)
        responses.append(response)
        if verbose:
            print(f"{step:2d}  {latency_ms:4.0f} ms  confidence {action.confidence:.2f}  "
                  f"{action.choice:<40} score {score}/{game_state.max_score}")
        if done:
            break
    env.close()

    return {
        "game": path.stem,
        "won": game_state.won,
        "lost": game_state.lost,
        "steps": step,
        "score": f"{score}/{game_state.max_score}",
        "median ms": round(statistics.median(latencies)),
        "max ms": round(max(latencies)),
        "over budget": sum(ms > LATENCY_BUDGET_MS for ms in latencies),
        "cost": f"${jev_cost(responses):.5f}",
    }


results = [play(games[0], verbose=True)]

Objective: You are hungry! Let's cook a delicious meal. Check the cookbook in the kitchen for the recipe. Once done, enjoy your meal!

 1   195 ms  confidence 0.98  go north                                 score 0/8
 2    96 ms  confidence 0.24  go west                                  score 0/8
 3   102 ms  confidence 0.66  examine cookbook                         score 0/8
 4   108 ms  confidence 0.42  take red potato from counter             score 1/8
 5    92 ms  confidence 0.14  take knife from counter                  score 1/8
 6   108 ms  confidence 0.97  chop red potato with knife               score 2/8
 7   119 ms  confidence 0.43  cook red potato with oven                score 3/8
 8    96 ms  confidence 0.51  take red bell pepper from counter        score 3/8
 9   104 ms  confidence 0.26  drop red bell pepper                     score 3/8
10   117 ms  confidence 0.30  examine counter                          score 3/8
11    98 ms  confidence 0.33  take red bell pepper     

In [22]:
results += [play(game) for game in games[1:]]
pd.DataFrame(results)

,game,won,lost,steps,score,median ms,max ms,over budget,cost
0,cooking-1,True,False,26,8/8,102,195,0,$0.00087
1,cooking-2,True,False,15,8/8,93,136,0,$0.00051
2,cooking-3,True,False,16,8/8,98,187,0,$0.00050
3,cooking-4,True,False,10,8/8,97,116,0,$0.00038
4,cooking-5,True,False,14,8/8,102,227,0,$0.00047
5,cooking-6,True,False,12,8/8,91,158,0,$0.00041
6,cooking-7,True,False,16,8/8,96,198,0,$0.00071
7,cooking-8,True,False,37,8/8,96,167,0,$0.00132
8,cooking-9,True,False,21,8/8,92,135,0,$0.00069
9,cooking-10,True,False,21,8/8,88,145,0,$0.00063


In [23]:
runs = pd.DataFrame(results)
decisions = runs["steps"].sum()
print(f"Won {runs['won'].sum()} of {len(runs)} games in {decisions} decisions; "
      f"{runs['over budget'].sum()} decisions over the {LATENCY_BUDGET_MS} ms budget")

Won 10 of 10 games in 188 decisions; 0 decisions over the 300 ms budget


In [24]:
results

[{'game': 'cooking-1',
  'won': True,
  'lost': False,
  'steps': 26,
  'score': '8/8',
  'median ms': 102,
  'max ms': 195,
  'over budget': 0,
  'cost': '$0.00087'},
 {'game': 'cooking-2',
  'won': True,
  'lost': False,
  'steps': 15,
  'score': '8/8',
  'median ms': 93,
  'max ms': 136,
  'over budget': 0,
  'cost': '$0.00051'},
 {'game': 'cooking-3',
  'won': True,
  'lost': False,
  'steps': 16,
  'score': '8/8',
  'median ms': 98,
  'max ms': 187,
  'over budget': 0,
  'cost': '$0.00050'},
 {'game': 'cooking-4',
  'won': True,
  'lost': False,
  'steps': 10,
  'score': '8/8',
  'median ms': 97,
  'max ms': 116,
  'over budget': 0,
  'cost': '$0.00038'},
 {'game': 'cooking-5',
  'won': True,
  'lost': False,
  'steps': 14,
  'score': '8/8',
  'median ms': 102,
  'max ms': 227,
  'over budget': 0,
  'cost': '$0.00047'},
 {'game': 'cooking-6',
  'won': True,
  'lost': False,
  'steps': 12,
  'score': '8/8',
  'median ms': 91,
  'max ms': 158,
  'over budget': 0,
  'cost': '$0.00041

In [25]:
games[0]

PosixPath('data/textworld/cooking-1.z8')

In [35]:
path = games[0]
env = textworld.start(str(path), request_infos=INFOS)
game_state = env.reset()
recipe, history, latencies, responses = None, [], [], []
print(f"Objective: {game_state.objective}\n")

MAX_STEPS = 2

for step in range(1, MAX_STEPS + 1):
    action, latency_ms, response = next_action(game_state, recipe, history)
    game_state, score, done = env.step(action.choice)

    print(action)

    if action.choice == "examine cookbook":
        recipe = clean(game_state.feedback)  # code keeps the plan
    history.append(action.choice)
    latencies.append(latency_ms)
    responses.append(response)
    if done:
        break
env.close()

Objective: You are hungry! Let's cook a delicious meal. Check the cookbook in the kitchen for the recipe. Once done, enjoy your meal!

type='choice' choice='go north' confidence=0.98 probabilities={'examine bed': 0.01, 'go north': 0.99}
type='choice' choice='go west' confidence=0.23 probabilities={'examine sofa': 0.02, 'go east': 0.35, 'go south': 0.21, 'go west': 0.42}


## 4. Verify Everything: Check Claims Against Evidence

Modeled on Jev MCP's [`jev_verify`](https://github.com/jkudish/jev-mcp#jev_verify) and TypeSafe's [citation-check cookbook](https://docs.typesafe.ai/cookbooks/citation_check). For each claim, Jev decides whether the evidence supports it, contradicts it, or says nothing about it. Confident verdicts stand; the rest go to review. This checks one supplied abstract rather than retrieving evidence or establishing scientific truth. “Auto” accepts the verdict, which may be contradicts or says_nothing; it does not mean the claim is true.

**Data:** [SciFact](https://github.com/allenai/scifact) via [`allenai/scifact_entailment`](https://huggingface.co/datasets/allenai/scifact_entailment), validation split: 340 claim–abstract pairs with labels from expert annotators. Claims and labels are CC BY 4.0; abstracts are ODC-By 1.0.

In [40]:
from datasets import load_dataset

claims = load_dataset("allenai/scifact_entailment", revision="refs/convert/parquet", split="validation").to_pandas()
claims["evidence"] = claims["abstract"].apply(" ".join)
claims["gold"] = claims["verdict"].map({"SUPPORT": "supports", "CONTRADICT": "contradicts", "NEI": "says_nothing"})

print(claims["gold"].value_counts().to_string(), "\n")
claims[["claim", "title", "gold"]].head()

gold
supports        138
says_nothing    131
contradicts      71 



,claim,title,gold
0,0-dimensional biomaterials show inductive prop...,New opportunities: the use of nanotechnologies...,says_nothing
1,"1,000 genomes project enables mapping of genet...",Rare Variants Create Synthetic Genome-Wide Ass...,supports
2,1/2000 in UK have abnormal PrP positivity.,Prevalent abnormal prion protein in human appe...,supports
3,5% of perinatal mortality is due to low birth ...,Estimates of global prevalence of childhood un...,says_nothing
4,A deficiency of vitamin B12 increases blood le...,Folic acid improves endothelial function in co...,says_nothing


### The Jev call: one `Choice` per claim

In [41]:
CLAIM_QUESTIONS = {
    "relation": Choice(
        instructions="How does `evidence` relate to `claim`?",
        criteria={
            "supports": "The evidence states the claim or directly implies that it is true",
            "contradicts": "The evidence states the opposite of the claim or implies that it is false",
            "says_nothing": "The evidence does not address what the claim asserts, either way",
        },
    ),
}

AUTO_ACCEPT = 0.8  # confidence needed for the verdict to stand without review

In [42]:
states = [{"claim": c.claim, "evidence": c.evidence} for c in claims.itertuples()]
responses, _ = await ask_all(states, CLAIM_QUESTIONS)

claims["verdict_jev"] = [r.answers["relation"].choice for r in responses]
claims["confidence"] = [r.answers["relation"].confidence for r in responses]
claims["probability"] = [r.answers["relation"].probabilities[r.answers["relation"].choice] for r in responses]
claims["action"] = ["auto" if c >= AUTO_ACCEPT else "review" for c in claims["confidence"]]
claims["correct"] = claims["verdict_jev"] == claims["gold"]

auto = claims[claims["action"] == "auto"]
print(f"Verdicts that stand: {len(auto) / len(claims):.0%} of claims, {auto['correct'].mean():.1%} correct")
print(f"Sent to review: {(claims['action'] == 'review').sum()}")
print(f"Accuracy if we trusted every verdict: {claims['correct'].mean():.1%}")

340 Jev calls in 3.0 s | median 94 ms per call | 257,564 input tokens | $0.0108
Verdicts that stand: 79% of claims, 90.3% correct
Sent to review: 72
Accuracy if we trusted every verdict: 84.1%


In [43]:
pd.crosstab(claims["gold"], claims["verdict_jev"], rownames=["gold"], colnames=["Jev"])

Jev,contradicts,says_nothing,supports
gold,,,
contradicts,64,3,4
says_nothing,15,97,19
supports,6,7,125


In [45]:
threshold_table(claims, correct="correct", confidence="confidence")

,threshold,handled automatically,samples,accuracy,errors
0,0.00,100%,340,84.1%,54
1,0.50,91%,311,87.8%,38
2,0.70,84%,286,88.8%,32
3,0.80,79%,268,90.3%,26
4,0.90,71%,242,90.9%,22
5,0.95,65%,221,91.0%,20
6,0.99,49%,166,94.0%,10
